# 实验 08：CIFAR-10 DCGAN
目标：理解 G / D 的交替更新、detach、固定噪声与训练不稳定。
参考：[PyTorch DCGAN 教程](https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html)。
本实验把卷积生成对抗网络改为 32×32 RGB 的 CIFAR-10，使用全部类别的混合图像，不输入类别标签。


In [ ]:
from pathlib import Path
import os, sys

# 从项目根目录、notebooks/ 或它们的子目录启动时都能找到 labkit。
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "labkit").is_dir()), None)
if ROOT is None:
    raise RuntimeError("请将整个课程目录解压后，从项目根目录启动 JupyterLab")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目根目录:", ROOT)


## 1. 数据加载
沿用相同的 CIFAR-10 划分，图像范围 [-1,1] 与生成器 Tanh 输出一致。


In [ ]:
from datetime import datetime
import inspect
import torch
from IPython.display import display, Image as DisplayImage
from labkit.common import choose_device, seed_everything, save_json
from labkit.data import load_image_data
from labkit.generative_models import ConvVAE, DCGenerator, DCDiscriminator, dcgan_init, vae_loss
from labkit.generative import (fit_vae, fit_gan, vae_epoch, gan_train_step, gan_diagnostics,
                               save_vae_examples, save_grid, noise, plot_generative_history)
SMOKE = os.environ.get("LAB_SMOKE", "0") == "1"
SEED, LATENT_DIM, CHANNELS, BATCH_SIZE = 42, 64, 32, 128
device = choose_device("auto")
seed_everything(SEED)
limits = [128, 64, 64] if SMOKE else [0, 0, 0]
# 与分类实验共享相同的 CIFAR-10 划分，但生成任务不做随机增广。
bundle = load_image_data(ROOT / "data", BATCH_SIZE, SEED, augment=False,
                         max_train=limits[0], max_val=limits[1], max_test=limits[2])
print("device:", device, "数据规模:", bundle.meta["sizes"], "SMOKE:", SMOKE)


## 2. 生成器与判别器
G 用转置卷积把噪声变成图像。D 输出单个 logit，使用 `BCEWithLogitsLoss`，无需手动 sigmoid。
D 的损失为真假两项的平均，G 的非饱和损失把生成图的目标设为 1。


In [ ]:
EPOCHS = 1 if SMOKE else 50
RUN_DIR = ROOT / "runs" / datetime.now().strftime("gan_%Y%m%d_%H%M%S_%f")
config = {"task": "gan", "model": "dcgan32", "epochs": EPOCHS, "lr": 0.0002,
          "latent_dim": LATENT_DIM, "channels": CHANNELS, "seed": SEED,
          "batch_size": BATCH_SIZE, "smoke": SMOKE,
          "torch_version": str(torch.__version__)}
generator = DCGenerator(LATENT_DIM, CHANNELS).to(device)
discriminator = DCDiscriminator(CHANNELS).to(device)
generator.apply(dcgan_init)
discriminator.apply(dcgan_init)
print(generator)
print(discriminator)


## 3. 一次完整的对抗更新
更新 D 时使用 `fake.detach()`，避免更新 G。更新 G 时冻结 D 的参数和 BN 状态，但保留 D 对输入的梯度。
G 与 D 使用各自独立的优化器。


In [ ]:
print(inspect.getsource(gan_train_step))
print(inspect.getsource(fit_gan))


## 4. 训练、验证诊断与固定噪声
每轮保存同一组噪声的生成图。使用预先指定的训练轮数，保存 last.pt，不按 D 的 loss 宣称某轮的 G 最好。


In [ ]:
history = fit_gan(generator, discriminator, bundle, device, RUN_DIR, config)
plot_generative_history(history, "gan", RUN_DIR / "learning_curves.png")
display(DisplayImage(filename=str(RUN_DIR / "previews" / f"epoch_{EPOCHS:03d}.png")))


## 5. 留出测试集诊断
这里记录 D 对未参与训练的真实图像和独立噪声生成图的分数，以及两者的通道均值/标准差。
**这些不是分类准确率或生成质量分数，不能替代 FID/KID 或覆盖率分析。**
本入门包不额外下载特征网络，也不把像素统计冒充 FID。比较图像质量需进一步规范评估协议。


In [ ]:
metrics = gan_diagnostics(generator, discriminator, bundle.test, device, LATENT_DIM, SEED + 2000)
save_json(RUN_DIR / "test_metrics.json", metrics)
print(metrics)
generator.eval()
with torch.no_grad():
    save_grid(generator(noise(64, LATENT_DIM, device, SEED + 3000)), RUN_DIR / "test_samples.png")
display(DisplayImage(filename=str(RUN_DIR / "test_samples.png")))
print(f'python generate.py --run-dir "{RUN_DIR}" --seed 2026')


## 6. 实操练习
1. 比较不同 epoch 的固定噪声图，记录清晰度、重复样本和类型覆盖。
2. 将 G 或 D 的学习率单独扩大，观察训练失衡。
3. 解释 `detach()` 和 `requires_grad_(False)` 的不同作用。
4. G loss 下降、D 输出接近 0.5，是否足以证明训练成功？给出原因。
5. 用相同噪声种子生成两次，确认恢复的模型给出相同结果。
